# Restaurant Sales Insight\n\nEnd-to-end analysis notebook for cleaning, exploratory data analysis, and portfolio-ready business insights.

In [ ]:
from pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nimport matplotlib.pyplot as plt\nimport seaborn as sns\n\nsns.set_theme(style='whitegrid', palette='Set2')\nplt.rcParams['figure.figsize'] = (10, 5)\n\nBASE_DIR = Path('..')\nDATA_PATH = BASE_DIR / 'data' / 'restaurant_sales_raw.csv'\nOUTPUT_PATH = BASE_DIR / 'output' / 'restaurant_sales_cleaned.csv'\n\ndf_raw = pd.read_csv(DATA_PATH)\ndf_raw.head()

## 1. Data Cleaning

In [ ]:
df = df_raw.copy()\ndf.columns = (\n    df.columns.str.strip()\n    .str.lower()\n    .str.replace(' ', '_', regex=False)\n)\n\ndf['date'] = pd.to_datetime(df['date'], format='%d-%m-%Y', errors='coerce')\ndf['price'] = pd.to_numeric(df['price'], errors='coerce')\ndf['quantity'] = pd.to_numeric(df['quantity'], errors='coerce')\ndf['product'] = df['product'].str.strip()\ndf['purchase_type'] = df['purchase_type'].str.strip()\ndf['payment_method'] = df['payment_method'].str.strip()\ndf['manager'] = df['manager'].str.replace(r'\\s+', ' ', regex=True).str.strip()\ndf['city'] = df['city'].str.strip()\n\ndf = df.dropna(subset=['date', 'product', 'price', 'quantity'])\ndf = df.drop_duplicates(subset=['order_id'])\n\ncategory_map = {\n    'Beverages': 'Beverage',\n    'Burgers': 'Main Course',\n    'Chicken Sandwiches': 'Main Course',\n    'Fries': 'Sides',\n    'Sides & Other': 'Sides',\n}\n\ndf['product_category'] = df['product'].map(category_map).fillna('Other')\ndf['revenue'] = (df['price'] * df['quantity']).round(2)\ndf['year'] = df['date'].dt.year\ndf['month_num'] = df['date'].dt.month\ndf['month_name'] = df['date'].dt.strftime('%b')\ndf['day_num'] = df['date'].dt.day\ndf['day_name'] = df['date'].dt.day_name()\ndf['week_num'] = df['date'].dt.isocalendar().week.astype(int)\ndf['is_weekend'] = df['day_name'].isin(['Saturday', 'Sunday'])\n\ndf = df[[\n    'order_id', 'date', 'year', 'month_num', 'month_name', 'week_num', 'day_num', 'day_name',\n    'is_weekend', 'product', 'product_category', 'price', 'quantity', 'revenue',\n    'purchase_type', 'payment_method', 'manager', 'city'\n]].sort_values(['date', 'order_id']).reset_index(drop=True)\n\ndf.to_csv(OUTPUT_PATH, index=False)\ndf.head()

In [ ]:
print('Rows:', len(df))\nprint('Columns:', len(df.columns))\nprint('Date range:', df['date'].min().date(), 'to', df['date'].max().date())\nprint('Total revenue:', round(df['revenue'].sum(), 2))\nprint('Total orders:', df['order_id'].nunique())\nprint('Average order value:', round(df['revenue'].sum() / df['order_id'].nunique(), 2))

## 2. Exploratory Data Analysis

In [ ]:
daily_sales = df.groupby('date', as_index=False)['revenue'].sum()\nmonthly_sales = df.groupby(['year', 'month_num', 'month_name'], as_index=False)['revenue'].sum()\nproduct_sales = df.groupby('product', as_index=False).agg(total_revenue=('revenue', 'sum'), total_quantity=('quantity', 'sum'))\nproduct_sales = product_sales.sort_values('total_revenue', ascending=False)\ncategory_sales = df.groupby('product_category', as_index=False)['revenue'].sum().sort_values('revenue', ascending=False)\ncity_sales = df.groupby('city', as_index=False)['revenue'].sum().sort_values('revenue', ascending=False)\n\nproduct_sales

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))\n\nsns.lineplot(data=daily_sales, x='date', y='revenue', marker='o', ax=axes[0, 0])\naxes[0, 0].set_title('Daily Revenue Trend')\naxes[0, 0].tick_params(axis='x', rotation=45)\n\nsns.barplot(data=product_sales, x='product', y='total_revenue', ax=axes[0, 1])\naxes[0, 1].set_title('Top Selling Menu by Revenue')\naxes[0, 1].tick_params(axis='x', rotation=20)\n\nsns.barplot(data=category_sales, x='product_category', y='revenue', ax=axes[1, 0])\naxes[1, 0].set_title('Revenue by Category')\n\nsns.barplot(data=city_sales, x='city', y='revenue', ax=axes[1, 1])\naxes[1, 1].set_title('Revenue by City')\naxes[1, 1].tick_params(axis='x', rotation=20)\n\nplt.tight_layout()\nplt.show()

In [ ]:
day_name_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']\nday_sales = df.groupby('day_name', as_index=False)['revenue'].sum()\nday_sales['day_name'] = pd.Categorical(day_sales['day_name'], categories=day_name_order, ordered=True)\nday_sales = day_sales.sort_values('day_name')\n\nplt.figure(figsize=(10, 5))\nsns.barplot(data=day_sales, x='day_name', y='revenue')\nplt.title('Revenue by Day of Week')\nplt.xticks(rotation=20)\nplt.show()

## 3. Trend Analysis and Business Notes\n\n- `Burgers` contributes the highest revenue and should remain the flagship menu.\n- `Main Course` dominates category contribution, while beverages can be improved through bundle strategies.\n- December revenue is higher than November, suggesting stronger year-end demand.\n- The source dataset does not include transaction hour timestamps, so a true `sales by hour` analysis is not possible without richer POS data.